# MISMIP+ marine ice sheet: two-layer composite rheology

This notebook sets up the MISMIP+ benchmark geometry (Asay-Davis et al. 2016)
with the multilayer dual model and solves the diagnostic momentum balance for a
composite, depth-varying rheology:

| Layer | Fraction | $n$ | $A$ (MPa$^{-n}$ yr$^{-1}$) | Mechanism |
|-------|----------|-----|----------------------------|-----------|
| Base  | 15%      | 4.0 | 46    | dislocation creep, temperate |
| Top   | 85%      | 1.8 | 0.24  | grain-boundary sliding, $-20$ degC |

The base rate factor is the temperate value of Goldsby & Kohlstedt (2001). The
top rate factor is the Cuffey & Paterson (2010) $n=3$ value at $-20$ degC
($1.2\times10^{-25}$ Pa$^{-3}$ s$^{-1}$ = $3.79$ MPa$^{-3}$ yr$^{-1}$),
stress-matched to $n=1.8$ at $\tau_{\rm ref}=0.1$ MPa:
$3.79\cdot0.1^{(3-1.8)}=0.24$.

The contrast in $A$ concentrates vertical shear near the bed, the physical
signature of a temperate basal layer. Grounded ice slides by a
regularised-Coulomb (Schoof) law whose drag vanishes at the grounding line,
and the terminus carries the ocean back-pressure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import firedrake
from firedrake import (
    Constant, Function, SpatialCoordinate, RectangleMesh,
    exp, sqrt, inner, dx, max_value, as_vector, derivative,
    DirichletBC, NonlinearVariationalProblem, NonlinearVariationalSolver,
)
from icepack2.constants import (
    ice_density as rho_I, water_density as rho_W, gravity as g,
)
from multilayer.model import minimization, utilities

In [ ]:
sparams = {
    "snes_type": "newtonls",
    "snes_max_it": 200,
    "snes_linesearch_type": "bt",
    "snes_divergence_tolerance": -1,
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
}
fcp = {"form_compiler_parameters": {"quadrature_degree": 6}}

# Boundary ids for RectangleMesh: 1 = divide (x=0), 2 = terminus (x=Lx),
# 3, 4 = side walls (y=0, y=Ly).
DIVIDE_ID, TERMINUS_ID, WALL_IDS = 1, 2, (3, 4)

## Domain and bed topography

The MISMIP+ domain is $640\times80$ km. The bed is a sixth-order polynomial in
$x$ with a Gaussian side-channel in $y$, floored at $-720$ m (Asay-Davis et al.
2016). It lies below sea level everywhere, so the ice sheet is marine.

In [ ]:
Lx, Ly = 640e3, 80e3
nx, ny = 128, 16
mesh = RectangleMesh(nx, ny, Lx, Ly)
x, y = SpatialCoordinate(mesh)
Q = firedrake.FunctionSpace(mesh, "CG", 1)

x_c = Constant(300e3)
X = x / x_c
B_0, B_2, B_4, B_6 = -150.0, -728.8, 343.91, -50.57
B_x = B_0 + B_2 * X**2 + B_4 * X**4 + B_6 * X**6

f_c, d_c, w_c = 4e3, 500.0, 24e3
B_y = d_c * (
    1 / (1 + exp(-2 * (y - Ly / 2 - w_c) / f_c))
    + 1 / (1 + exp(+2 * (y - Ly / 2 + w_c) / f_c))
)
bed = Function(Q, name="bed").interpolate(max_value(B_x + B_y, Constant(-720.0)))

fig, ax = plt.subplots(figsize=(10, 2.4))
tpc = firedrake.tripcolor(bed, axes=ax, cmap="terrain")
ax.set_title("MISMIP+ bed elevation (m)")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
fig.colorbar(tpc, ax=ax);

## Initial geometry

We prescribe a smooth grounded profile that thins seaward into a floating shelf.
The surface follows flotation, and the effective pressure
$N=\rho_I g\,(H-H_f)$ (with flotation thickness $H_f=-b\,\rho_W/\rho_I$) is
clamped at zero, so it vanishes on floating ice. The grounding line sits where
the height above flotation $H-H_f$ crosses zero.

(This prescribed thickness is an initial condition for the diagnostic solve;
the true MISMIP+ steady state is reached by the prognostic spin-up in
`mismip_plus_spinup.py`.)

In [ ]:
R_FLOT = float(rho_I / rho_W)
H_div, H_min = 2000.0, 50.0
H = Function(Q, name="thickness").interpolate(
    max_value(Constant(H_min), Constant(H_div) * max_value(Constant(0.0), 1 - x / Lx) ** 0.5)
)
surface = Function(Q, name="surface").interpolate(
    max_value(bed + H, (1 - Constant(R_FLOT)) * H)
)
H_f = max_value(Constant(0.0), -bed) * Constant(rho_W / rho_I)
N = Function(Q, name="effective_pressure").interpolate(
    max_value(rho_I * g * (H - H_f), Constant(0.0))
)

# Centreline section
xs = np.linspace(0.0, Lx, 201)
b_line = np.array([bed.at([xi, Ly / 2]) for xi in xs])
s_line = np.array([surface.at([xi, Ly / 2]) for xi in xs])
h_line = np.array([H.at([xi, Ly / 2]) for xi in xs])
base_line = s_line - h_line
haf_line = h_line - np.maximum(0.0, -b_line) / R_FLOT
gl = xs[np.where((haf_line[:-1] > 0) & (haf_line[1:] <= 0))[0]]
x_gl = gl[0] if len(gl) else None

fig, ax = plt.subplots(figsize=(9, 3))
ax.fill_between(xs / 1e3, b_line, -1500, color="0.6", label="bedrock")
ax.fill_between(xs / 1e3, base_line, s_line, color="powderblue", label="ice")
ax.axhline(0.0, color="navy", ls=":", lw=1, label="sea level")
if x_gl is not None:
    ax.axvline(x_gl / 1e3, color="red", ls="--", label=f"grounding line ({x_gl/1e3:.0f} km)")
ax.set_xlabel("x (km)"); ax.set_ylabel("elevation (m)"); ax.set_ylim(-1500, 2200)
ax.set_title("Centreline section (y = 40 km)"); ax.legend(fontsize=8);

## Composite rheology

Both layers use $n$ as a `Constant` so we can ramp it during the solve. The
targets are the composite values; the warm start (below) begins from the
effective Glen $n=3$ rheology, which is the depth-average of $n=4$ and
$n=1.8$.

In [ ]:
LAYER_FRACTIONS = [0.15, 0.85]
LAYER_EXPONENTS = [4.0, 1.8]
LAYER_PREFACTORS = [46.0, 0.24]
NUM_LAYERS = 2

# Depth-averaged ("effective Glen") rheology for the warm start.
N_AVG, A_AVG = 3.0, 100.0

# Regularised-Coulomb basal sliding: beta^2 = C * N
COULOMB_C, U_0, SLIDING_M = 0.5, 300.0, 3.0

## Depth-averaged warm start

A single-layer (SSA-like) diagnostic solve with one effective rate factor
provides the initial velocity for the two-layer model. We ramp the exponent
from $1$ to $3$ for robustness.

In [ ]:
beta2 = COULOMB_C * N

Z1 = utilities.create_function_space(mesh, 1)
z1 = Function(Z1)
z1.sub(0).interpolate(as_vector([Constant(10.0), Constant(0.0)]))
f1 = utilities.split_fields(firedrake.split(z1), 1)[0]

n1 = Constant(1.0)
A1 = Constant(A_AVG)

L1 = (
    minimization.viscous_power(
        membrane_stress=f1["membrane_stress"], thickness=H,
        flow_law_coefficient=A1, flow_law_exponent=n1,
    )
    + minimization.momentum_balance(
        velocity=f1["velocity"], membrane_stress=f1["membrane_stress"],
        thickness=H, surface=surface, basal_stress=f1["interlayer_stress"],
    )
    + minimization.schoof_friction_power(
        basal_stress=f1["interlayer_stress"], friction_coefficient=beta2,
        transition_speed=Constant(U_0), sliding_exponent=Constant(SLIDING_M),
    )
    + minimization.calving_terminus(
        velocity=f1["velocity"], thickness=H, surface=surface,
        outflow_ids=(TERMINUS_ID,), layer_fraction=Constant(1.0),
    )
)
F1 = derivative(L1, z1)
bcs1 = [
    DirichletBC(Z1.sub(0).sub(0), 0, (DIVIDE_ID,)),
    DirichletBC(Z1.sub(0).sub(1), 0, WALL_IDS),
]
problem1 = NonlinearVariationalProblem(F1, z1, bcs1, **fcp)
solver1 = NonlinearVariationalSolver(problem1, solver_parameters=sparams)

for lam in np.linspace(0.0, 1.0, 9):
    n1.assign((1 - lam) + lam * N_AVG)
    solver1.solve()

warm_u = z1.subfunctions[0].copy(deepcopy=True)
print(f"max depth-averaged speed: {np.max(np.abs(warm_u.dat.data_ro[:, 0])):.1f} m/yr")

## Two-layer solve

We build the multilayer action functional (viscous power per layer, momentum
balance, interlayer shear, basal Schoof drag, and the calving terminus), warm
start every layer's velocity, and run a continuation that relaxes the per-layer
exponents from $3$ to the $n=4$ / $n=1.8$ targets while splitting the rate
factor in log-space.

In [ ]:
h_layers = utilities.layer_thicknesses(H, NUM_LAYERS, fractions=LAYER_FRACTIONS)
n_consts = [Constant(N_AVG) for _ in range(NUM_LAYERS)]
A_consts = [Constant(A_AVG) for _ in range(NUM_LAYERS)]

Z = utilities.create_function_space(mesh, NUM_LAYERS)
z = Function(Z)
for l in range(NUM_LAYERS):
    z.sub(3 * l).interpolate(warm_u)
fields = utilities.split_fields(firedrake.split(z), NUM_LAYERS)

Lag = Constant(0) * dx(mesh)
for l in range(NUM_LAYERS):
    h_l, fl = h_layers[l], fields[l]
    S_above = fields[l + 1]["interlayer_stress"] if l < NUM_LAYERS - 1 else None
    Lag += minimization.viscous_power(
        membrane_stress=fl["membrane_stress"], thickness=h_l,
        flow_law_coefficient=A_consts[l], flow_law_exponent=n_consts[l],
    )
    Lag += minimization.momentum_balance(
        velocity=fl["velocity"], membrane_stress=fl["membrane_stress"],
        thickness=h_l, surface=surface,
        basal_stress=fl["interlayer_stress"] if l == 0 else None,
        stress_above=S_above,
        stress_below=fl["interlayer_stress"] if l > 0 else None,
    )
    Lag += minimization.calving_terminus(
        velocity=fl["velocity"], thickness=H, surface=surface,
        outflow_ids=(TERMINUS_ID,), layer_fraction=h_l / H,
    )

Lag += minimization.schoof_friction_power(
    basal_stress=fields[0]["interlayer_stress"], friction_coefficient=beta2,
    transition_speed=Constant(U_0), sliding_exponent=Constant(SLIDING_M),
)
for l in range(1, NUM_LAYERS):
    Lag += minimization.interlayer_power(
        interlayer_stress=fields[l]["interlayer_stress"],
        thickness_above=h_layers[l], thickness_below=h_layers[l - 1],
        flow_law_coefficient=A_consts[l - 1], flow_law_exponent=n_consts[l - 1],
    )

F = derivative(Lag, z)
bcs = []
for l in range(NUM_LAYERS):
    bcs.append(DirichletBC(Z.sub(3 * l).sub(0), 0, (DIVIDE_ID,)))
    bcs.append(DirichletBC(Z.sub(3 * l).sub(1), 0, WALL_IDS))

problem = NonlinearVariationalProblem(F, z, bcs, **fcp)
solver = NonlinearVariationalSolver(problem, solver_parameters=sparams)

lambdas = np.concatenate([
    np.linspace(0.0, 0.7, 8),
    np.linspace(0.7, 0.92, 8)[1:],
    np.linspace(0.92, 1.0, 7)[1:],
])
for lam in lambdas:
    for l in range(NUM_LAYERS):
        n_consts[l].assign((1 - lam) * N_AVG + lam * LAYER_EXPONENTS[l])
        A_consts[l].assign(np.exp((1 - lam) * np.log(A_AVG) + lam * np.log(LAYER_PREFACTORS[l])))
    solver.solve()
print("two-layer solve complete")

## The solution

We compare the surface (top, $n=1.8$) and basal (bottom, $n=4$ temperate) speeds
along the centreline. The difference is the vertical shear, which the temperate
base concentrates. We also map the surface speed with the grounding line
overlaid.

In [ ]:
u_bot = z.subfunctions[0]
u_top = z.subfunctions[3]
s_top = np.array([u_top.at([xi, Ly / 2])[0] for xi in xs])
s_bot = np.array([u_bot.at([xi, Ly / 2])[0] for xi in xs])

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(xs / 1e3, s_top, "C0-", lw=2, label="surface (top, n=1.8)")
ax.plot(xs / 1e3, s_bot, "C1-", lw=2, label="base (bottom, n=4 temperate)")
ax.fill_between(xs / 1e3, s_bot, s_top, color="C0", alpha=0.15, label="vertical shear")
if x_gl is not None:
    ax.axvline(x_gl / 1e3, color="red", ls="--", lw=1, label="grounding line")
ax.set_xlabel("x (km)"); ax.set_ylabel("along-flow speed (m/yr)")
ax.set_title("MISMIP+ centreline speeds"); ax.legend(fontsize=8);

In [ ]:
speed = Function(Q, name="surface_speed").interpolate(sqrt(inner(u_top, u_top)))
haf = Function(Q).interpolate(H - max_value(Constant(0.0), -bed) * Constant(rho_W / rho_I))

fig, ax = plt.subplots(figsize=(10, 2.6))
tpc = firedrake.tripcolor(speed, axes=ax, cmap="viridis")
firedrake.tricontour(haf, levels=[0.0], axes=ax, colors="red")
ax.set_title("Surface speed (m/yr) with grounding line (red)")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
fig.colorbar(tpc, ax=ax);

## Next step: prognostic spin-up

The diagnostic solve above uses a fixed, prescribed thickness. To reach a true
MISMIP+ steady state and study grounding-line response, evolve the thickness
with the depth-integrated continuity equation driven by the thickness-weighted
depth-averaged velocity
$\bar u = (h_1 u_1 + h_2 u_2)/H$. That spin-up, together with the MISMIP+ Ice1r
sub-shelf melt forcing, is implemented in `mismip_plus_spinup.py`.